# Fine-tune the PaddleOCR recognition model on Penn plan text lines

Runs on a Colab GPU. Before starting: **Runtime > Change runtime type > T4 GPU**.

You need one file from the project: `work/colab/penn_ocr_finetune.zip`
(training and validation line images with human transcriptions, plus the scripts).
The test partition is not in it.

Run the cells top to bottom. The last cell downloads `penn_finetuned.zip`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Install PaddlePaddle (GPU build, same version as the local baseline)

In [ ]:
!pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
!pip install -q pyclipper lmdb rapidfuzz albumentations scikit-image shapely pyyaml
import paddle
paddle.utils.run_check()
print(paddle.__version__, "GPU:", paddle.device.is_compiled_with_cuda())

## 2. Upload `penn_ocr_finetune.zip`

In [ ]:
import os, zipfile
from google.colab import files
os.makedirs("/content/penn", exist_ok=True)
os.chdir("/content/penn")
if not os.path.exists("work/rec_dataset/rec_gt_train.txt"):
    up = files.upload()                       # choose penn_ocr_finetune.zip
    zipfile.ZipFile(next(iter(up))).extractall(".")
print(sum(1 for _ in open("work/rec_dataset/rec_gt_train.txt")), "training lines,",
      sum(1 for _ in open("work/rec_dataset/rec_gt_val.txt")), "validation lines")

## 3. Train

The script clones PaddleOCR v3.7.0, downloads the pretrained PP-OCRv6 weights,
scores the pretrained model on the validation lines, fine-tunes, and scores the
best checkpoint again. Change `--epochs` / `--lr` here if you want another run.

In [ ]:
%cd /content/penn
!bash finetune/run_finetune.sh --gpu --epochs 20 --lr 2e-5 --batch-size 8

## 4. Before / after on the validation lines

In [ ]:
import re
for name in ("eval_before", "eval_after"):
    log = open(f"/content/penn/work/finetune/{name}.log").read()
    print(name, {k: float(v) for k, v in re.findall(r"(acc|norm_edit_dis):([0-9.]+)", log)})

## 5. Download the fine-tuned model

In [ ]:
%cd /content/penn/work/finetune
!zip -q -r /content/penn_finetuned.zip inference config.yml train.log eval_before.log eval_after.log
files.download("/content/penn_finetuned.zip")

Back on the Mac, unzip it into `work/finetune/` so that `work/finetune/inference/` exists, then:

```
python -m penn_ocr.predict --engine paddle --rec-model-dir work/finetune/inference --name paddle_finetuned
python -m penn_ocr.evaluate --split val
python -m penn_ocr.evaluate --split test
```